## Gold Layer

Building gold layer based on dimensional model. Creating the fct_results, dim_event, dim_athelete, and views for each marathon typ.

In [0]:
from pyspark.sql.functions import col
# reading in from silver OBT
df = spark.read.table("marathos.silver.races_obt")
print(f"Rows loaded from the silver:{df.count()}")

## Fact Table - fct_results
Here i am creating the central fact table with the performance metrics and foregin keys to the dimensions. 

In [0]:
# Creating fct_results - central fact table 

fct_results = df.select(
    col("result_id"),
    col("event_id"),
    col("athlete_id"),
    col("year_of_event"),
    col("athlete_performance"),
    col("performance_seconds"),
    col("performance_km"),
    col("athlete_average_speed")
)
fct_results.write.format("delta").saveAsTable("marathos.gold.fct_results")
print(f"fct_results written: {fct_results.count()} rows")

## Dimension Table - dim_event
Creating the event dimension with unique events and attributes

In [0]:
# CReating dim_event - Unique events only 

dim_event = df.select(
    col("event_id"),
    col("event_name"),
    col("event_distance_length"),
    col("event_number_of_finishers"),
    col("event_dates"),
    col("year_of_event")
).dropDuplicates(["event_id"])

dim_event.write.format("delta").saveAsTable("marathos.gold.dim_event")
print(f"dim_event written: {dim_event.count()} rows")


## Dimension Table - dim_athlete
creating athele dimension with unique athletes only 

In [0]:
# Create dim_athlete - unique athletes only
dim_athlete = df.select(
    col("athlete_id"),
    col("athlete_country"),
    col("athlete_year_of_birth"),
    col("athlete_gender"),
    col("athlete_age_category"),
    col("athlete_club")
).dropDuplicates(["athlete_id"])

dim_athlete.write.format("delta").saveAsTable("marathos.gold.dim_athlete")
print(f"dim_athlete written: {dim_athlete.count()} rows")